# Building Custom Tasks

This notebook shows how to:
- Create a custom gridworld task
- Define task objectives and rewards
- Register your task
- Test and evaluate custom tasks

## Setup

In [ ]:
import gymnasium as gym

from agentick.core.entity import Entity, EntityType
from agentick.core.env import GridWorldEnv
from agentick.tasks.registry import register_task

## Define a Custom Task

Let's create a "Treasure Hunt" task where the agent must collect coins before reaching the goal:

In [ ]:
class TreasureHuntEnv(GridWorldEnv):
    """
    Custom task: Agent must collect at least 3 coins before reaching the goal.

    Success criteria:
    - Agent reaches goal
    - Agent has collected at least 3 coins

    Entities:
    - Agent (blue)
    - Goal (green)
    - Coins (yellow)
    - Walls (gray)
    """

    def __init__(self, difficulty="easy", **kwargs):
        """Initialize treasure hunt environment."""
        # Set grid size based on difficulty
        size_map = {"easy": 6, "medium": 8, "hard": 10, "expert": 12}
        grid_size = size_map.get(difficulty, 6)

        super().__init__(grid_size=grid_size, max_steps=100, **kwargs)

        self.difficulty = difficulty
        self.coins_collected = 0
        self.min_coins_required = 3

    def reset(self, seed=None, options=None):
        """Reset environment and generate new level."""
        super().reset(seed=seed)

        # Reset coin counter
        self.coins_collected = 0

        # Clear grid
        self.grid.clear()

        # Place agent in bottom-left
        self.agent_pos = (1, self.grid.height - 2)
        self.grid.set(self.agent_pos[0], self.agent_pos[1], Entity(EntityType.AGENT, color="blue"))

        # Place goal in top-right
        goal_pos = (self.grid.width - 2, 1)
        self.grid.set(goal_pos[0], goal_pos[1], Entity(EntityType.GOAL, color="green"))

        # Place coins
        num_coins = 5 if self.difficulty in ["easy", "medium"] else 7
        for _ in range(num_coins):
            x = self.np_random.integers(1, self.grid.width - 1)
            y = self.np_random.integers(1, self.grid.height - 1)

            # Don't place on agent or goal
            if (x, y) != self.agent_pos and (x, y) != goal_pos:
                self.grid.set(x, y, Entity(EntityType.KEY, color="yellow"))

        # Place some walls
        num_walls = 3 if self.difficulty == "easy" else 8
        for _ in range(num_walls):
            x = self.np_random.integers(1, self.grid.width - 1)
            y = self.np_random.integers(1, self.grid.height - 1)

            # Don't block important spots
            if (x, y) != self.agent_pos and (x, y) != goal_pos:
                current = self.grid.get(x, y)
                if current is None:
                    self.grid.set(x, y, Entity(EntityType.WALL, color="gray"))

        obs = self._get_obs()
        info = self._get_info()

        return obs, info

    def step(self, action):
        """Take a step in the environment."""
        # Get new position
        dx, dy = self._action_to_direction(action)
        new_x = self.agent_pos[0] + dx
        new_y = self.agent_pos[1] + dy

        # Check if move is valid
        if not self.grid.in_bounds(new_x, new_y):
            # Hit wall, small penalty
            reward = -0.01
            terminated = False
        else:
            entity = self.grid.get(new_x, new_y)

            if entity and entity.type == EntityType.WALL:
                # Hit wall
                reward = -0.01
                terminated = False
            elif entity and entity.type == EntityType.KEY:
                # Collect coin!
                self.coins_collected += 1
                reward = 0.1
                terminated = False

                # Move agent and remove coin
                self.grid.set(self.agent_pos[0], self.agent_pos[1], None)
                self.agent_pos = (new_x, new_y)
                self.grid.set(new_x, new_y, Entity(EntityType.AGENT, color="blue"))
            elif entity and entity.type == EntityType.GOAL:
                # Reached goal - check if we have enough coins
                if self.coins_collected >= self.min_coins_required:
                    # Success!
                    reward = 1.0
                    terminated = True
                else:
                    # Reached goal but not enough coins - fail
                    reward = -0.5
                    terminated = True

                # Move to goal
                self.grid.set(self.agent_pos[0], self.agent_pos[1], None)
                self.agent_pos = (new_x, new_y)
                self.grid.set(new_x, new_y, Entity(EntityType.AGENT, color="blue"))
            else:
                # Empty space - valid move
                reward = -0.01  # Small step penalty
                terminated = False

                # Move agent
                self.grid.set(self.agent_pos[0], self.agent_pos[1], None)
                self.agent_pos = (new_x, new_y)
                self.grid.set(new_x, new_y, Entity(EntityType.AGENT, color="blue"))

        # Increment step counter
        self.step_count += 1

        # Check if max steps reached
        truncated = self.step_count >= self.max_steps

        obs = self._get_obs()
        info = self._get_info()
        info["coins_collected"] = self.coins_collected
        info["success"] = terminated and reward > 0

        return obs, reward, terminated, truncated, info

    def _action_to_direction(self, action):
        """Convert action to direction."""
        # 0: up, 1: right, 2: down, 3: left
        directions = [(0, -1), (1, 0), (0, 1), (-1, 0)]
        return directions[action]

    def _get_obs(self):
        """Get current observation."""
        if self.render_mode == "ascii":
            return self.grid.render_ascii()
        elif self.render_mode == "rgb_array":
            return self.grid.render_rgb()
        else:
            return self.grid.to_numpy()

    def _get_info(self):
        """Get info dict."""
        return {
            "step_count": self.step_count,
            "agent_pos": self.agent_pos,
            "coins_collected": self.coins_collected,
        }


print("Custom TreasureHuntEnv defined!")

## Register the Custom Task

Register it so it can be used like any other Agentick task:

In [ ]:
# Register with gymnasium
gym.register(
    id="TreasureHunt-v0",
    entry_point=lambda **kwargs: TreasureHuntEnv(**kwargs),
)

# Also register with agentick
register_task(
    task_id="TreasureHunt-v0",
    task_class=TreasureHuntEnv,
    description="Collect coins before reaching the goal",
    capabilities=["navigation", "planning"],
)

print("✓ Custom task registered as TreasureHunt-v0")

## Test the Custom Task

Let's test our custom task:

In [ ]:
# Create environment
env = gym.make("TreasureHunt-v0", difficulty="easy", render_mode="ascii")

# Reset and show initial state
obs, info = env.reset(seed=42)

print("Initial State:")
print(obs)
print(f"\nCoins collected: {info['coins_collected']}")
print(f"Agent position: {info['agent_pos']}")

In [ ]:
# Run a random episode
print("\nRunning random agent...\n")

obs, info = env.reset(seed=123)
total_reward = 0
max_coins = 0

for step in range(100):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)

    total_reward += reward
    max_coins = max(max_coins, info["coins_collected"])

    if terminated or truncated:
        break

print("Episode complete!")
print(f"Steps: {step + 1}")
print(f"Total reward: {total_reward:.3f}")
print(f"Coins collected: {max_coins}")
print(f"Success: {info.get('success', False)}")

print("\nFinal state:")
print(obs)

env.close()

## Evaluate on Multiple Episodes

Test performance across several episodes:

In [ ]:
import matplotlib.pyplot as plt

env = gym.make("TreasureHunt-v0", difficulty="easy", render_mode="ascii")

results = []
n_episodes = 20

print(f"Running {n_episodes} episodes...\n")

for episode in range(n_episodes):
    obs, info = env.reset(seed=1000 + episode)
    total_reward = 0
    max_coins = 0

    for step in range(100):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)

        total_reward += reward
        max_coins = max(max_coins, info["coins_collected"])

        if terminated or truncated:
            break

    success = info.get("success", False)
    results.append(
        {
            "episode": episode,
            "reward": total_reward,
            "coins": max_coins,
            "steps": step + 1,
            "success": success,
        }
    )

    print(
        f"Episode {episode + 1:2d}: reward={total_reward:6.3f}, coins={max_coins}, steps={step + 1:3d}, success={success}"
    )

env.close()

# Summary statistics
import pandas as pd

df = pd.DataFrame(results)

print("\n" + "=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)
print(f"Success rate: {df['success'].mean():.1%}")
print(f"Average reward: {df['reward'].mean():.3f}")
print(f"Average coins: {df['coins'].mean():.1f}")
print(f"Average steps: {df['steps'].mean():.1f}")

In [ ]:
# Visualize results
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))

# Rewards over episodes
ax1.plot(df["episode"], df["reward"], marker="o", linewidth=2, markersize=6)
ax1.set_xlabel("Episode")
ax1.set_ylabel("Total Reward")
ax1.set_title("Reward per Episode")
ax1.grid(True, alpha=0.3)

# Coins collected
ax2.bar(df["episode"], df["coins"], color="gold", alpha=0.7)
ax2.axhline(y=3, color="red", linestyle="--", label="Required (3)")
ax2.set_xlabel("Episode")
ax2.set_ylabel("Coins Collected")
ax2.set_title("Coins Collected per Episode")
ax2.legend()
ax2.grid(True, alpha=0.3)

# Success/failure distribution
success_counts = df["success"].value_counts()
ax3.pie(
    success_counts,
    labels=["Failure", "Success"],
    autopct="%1.1f%%",
    colors=["#ff6b6b", "#51cf66"],
    startangle=90,
)
ax3.set_title("Success vs Failure")

# Steps distribution
ax4.hist(df["steps"], bins=15, color="steelblue", alpha=0.7, edgecolor="black")
ax4.axvline(
    df["steps"].mean(),
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Mean: {df['steps'].mean():.1f}",
)
ax4.set_xlabel("Steps")
ax4.set_ylabel("Frequency")
ax4.set_title("Episode Length Distribution")
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Using Your Custom Task in Experiments

You can now use your custom task like any other Agentick task:

In [ ]:
from agentick.experiments import ExperimentConfig, ExperimentRunner

# Create experiment config with custom task
config = ExperimentConfig(
    name="treasure_hunt_eval",
    agent={"type": "random"},
    tasks=["TreasureHunt-v0"],  # Use our custom task!
    difficulties=["easy"],
    n_episodes=10,
    n_seeds=3,
    render_modes=["ascii"],
    output_dir="results/custom_task",
)

# Run experiment
print("Running experiment with custom task...")
runner = ExperimentRunner(config)
results = runner.run()

print("\nExperiment complete!")
print(f"Results saved to: {config.output_dir}")

## Key Takeaways

You've learned how to:

1. **Define custom environments** by extending `GridWorldEnv`
2. **Implement custom logic** for rewards, termination, entities
3. **Register tasks** to use them throughout Agentick
4. **Test and evaluate** your custom tasks
5. **Integrate with experiments** for systematic evaluation

## Next Steps

- Create more complex custom tasks
- Define custom observation spaces
- Implement custom rendering modes
- Share your tasks with the community!
- Train RL agents on your custom tasks